# MediStock AI — Exploratory Data Analysis

This notebook is a thin, readable wrapper around the reusable EDA module
(`ml/evaluation/eda.py`). The analysis lives in code so it is testable and repeatable;
the notebook exists to read it interactively and to attach business interpretation.

Run from the repository root. Prerequisite: the pipeline has produced
`data/processed/*.parquet` and (optionally) `artifacts/v1/`.

All data is **synthetic** — patterns below describe the simulator, not a real pharmacy.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from app.core.config import get_settings
from ml.common.io import read_parquet

settings = get_settings()
processed = settings.processed_data_dir

products = read_parquet(processed / "products.parquet")
sales = read_parquet(processed / "sales.parquet")
inventory = read_parquet(processed / "inventory.parquet")
batches = read_parquet(processed / "batches.parquet")
panel = read_parquet(processed / "demand_panel.parquet")

print(f"products={len(products):,} sales={len(sales):,} batches={len(batches):,} panel={len(panel):,}")

## 1. Demand scale by category

**Business question:** which categories carry the volume, and therefore where a forecast
error is most expensive?

Analgesics and vitamins dominate. That ordering is what the generator's category baselines
encode, and it is why `category` is a model feature.

In [ ]:
merged = panel.merge(products[["product_id", "category"]], on="product_id")
summary = (
    merged.groupby("category")["quantity"]
    .agg(total="sum", mean_per_day="mean", sd="std")
    .sort_values("total", ascending=False)
)
summary.assign(share=summary["total"] / summary["total"].sum()).round(3)

## 2. Weekly and annual structure

**Business question:** does demand depend on the day of week and the season?

If yes, calendar features at the prediction origin are justified. Clinic-driven categories
dip at weekends while OTC categories rise — a single global weekday multiplier would blur
this, which is why the simulator applies category-specific profiles.

In [ ]:
sales_cat = sales.merge(products[["product_id", "category"]], on="product_id")

weekday = (
    sales_cat.groupby(["category", "day_of_week"])["quantity_sold"].mean().unstack()
)
weekday.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
weekday.round(1)

In [ ]:
monthly = panel.assign(month=panel["date"].dt.month).groupby("month")["quantity"].mean()
monthly.round(1).to_frame("mean_units_per_product_day")

## 3. Zero-demand structure

**Business question:** how common are zero-sales days, and does that break our metrics?

Zero days are common and highly skewed toward a subset of slow movers. This is the reason
MAPE is unusable (undefined at zero) and WAPE is the headline metric, and why the serving
layer flags sparse products with low confidence.

In [ ]:
per_product = (
    panel.assign(zero=(panel["quantity"] == 0).astype(int))
    .groupby("product_id")["zero"]
    .mean()
)
print(f"zero-day share overall: {(panel['quantity'] == 0).mean():.2%}")
print(f"products with >50% zero days: {(per_product > 0.5).sum()}")
per_product.describe().round(3)

## 4. Inventory posture and expiry windows

**Business question:** do we have both over-stocking and under-stocking at the same time,
and how much stock expires soon?

Both failure modes coexist, which is why the recommendation layer must report them
separately rather than collapsing to one "inventory health" score.

In [ ]:
snapshot = pd.Timestamp("2025-10-01")
batches["days_to_expiry"] = (pd.to_datetime(batches["expiry_date"]) - snapshot).dt.days

bins = [-1000, 0, 30, 60, 90, 180, 365, 10000]
labels = ["expired", "0-30", "31-60", "61-90", "91-180", "181-365", ">365"]
buckets = pd.cut(batches["days_to_expiry"], bins=bins, labels=labels)
batches.groupby(buckets, observed=True)["batch_id"].count().to_frame("batches")

In [ ]:
# Coverage needs the forecast; fall back to trailing demand if no artifact exists yet.
from app.services.artifact_store import ArtifactStore
from app.services.forecast_service import ForecastService

store = ArtifactStore(settings=settings)
if store.is_ready():
    forecast = ForecastService(store.load()).forecast_all()[["product_id", "predicted_demand"]]
else:
    forecast = (
        panel[panel["date"] > snapshot - pd.Timedelta(days=30)]
        .groupby("product_id")["quantity"].sum().rename("predicted_demand").reset_index()
    )

coverage = inventory.merge(forecast, on="product_id")
coverage = coverage[coverage["predicted_demand"] > 0]
coverage["coverage_days"] = coverage["current_stock"] / (coverage["predicted_demand"] / 30)
coverage["coverage_days"].describe().round(1)

## 5. Regenerate the full figure set

The notebook is for exploration; the deliverable figures and the written report are
produced by the pipeline command below (it writes `reports/figures/*.png` and
`reports/eda_report.md`).

In [ ]:
from ml.evaluation.eda import run_eda

report = run_eda(
    products=products,
    sales=sales,
    inventory=inventory,
    batches=batches,
    panel=panel,
    forecast=forecast,
    snapshot="2025-10-01",
    figures_dir=settings.figures_dir,
    report_path=settings.reports_dir / "eda_report.md",
)
print(f"wrote {report}")
print(f"figures written: {len(list(settings.figures_dir.glob('*.png')))}")